# Trade Analysis
This notebook analyzes the results of the grid search experiment, focusing on the trade lists generated.
It covers:
1. Loading Trade Data
2. Calculating Key Performance Metrics
3. Visualizing Equity Curves
4. Analyzing Trade Distribution and Duration
5. Evaluating Drawdown Characteristics
6. Comparing Strategy Performance by Asset
7. Analyzing Win Rate vs. Risk-Reward Ratio

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Try 'notebook' renderer (offline mode) and force a dark template
# This embeds the Plotly JS library directly in the notebook
pio.renderers.default = "notebook"
pio.templates.default = "plotly_dark"

import glob
import os


In [2]:
import sys
# Add project root to path to import src modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.trading_strategy.utils.helpers import is_outlier, is_outlier_mod_zscore, calculate_trade_statistics

## 1. Load Trade Data
Load trade CSV files. This section accepts a list of paths which can be:
- Local file paths (e.g., `../data/trades/trades_BTC.csv`)
- MLflow artifact URIs (e.g., `runs:/<run_id>/trades.csv`)

If the list is empty, it defaults to loading all CSV files from `../data/trades/`.

In [13]:
import mlflow
import urllib.parse
import re

# Set MLflow tracking URI to the database in the project root
# We assume the notebook is in 'examples/' and 'mlflow.db' is in the parent directory
db_path = os.path.abspath(os.path.join(os.getcwd(), "..", "mlflow.db"))
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

# List of paths to trade files. 
# Can be local paths, MLflow artifact URIs (e.g., "runs:/<run_id>/trades.csv"), file URIs,
# OR directly a Run ID (e.g. "6ed278fe535c41958baa964d968ba132")
trade_sources = [
    # Example:
    # "runs:/00050a99d2924d078f8ee52bced48a34/trades_BTCUSDT_1h_rank1.csv",
    "86a5affeeeb248b1b18a3a03e8d9781b",
    "ca3981e36ef94f859e090a6423fdd405",
    "eabe859be4d64284955d36b4f2fe8738",
    "31b3d21e7b07478a98cd7198e2e051d7",
    "eb913bf9baf54fb69513a88759653bfa",
    "3548a88b870848028fd87ed9809dfe16",
    "6fd43ddd3c094232bf70f4e3b7763fc6",
    "42edb4d61c8a4e10af687b2eacc3dbfb"
]

# If list is empty, default to globbing the local directory
if not trade_sources:
    trades_path = "../data/trades/"
    trade_sources = glob.glob(os.path.join(trades_path, "trades_*.csv"))

print(f"Processing {len(trade_sources)} trade sources.")

all_trades = []

for source in trade_sources:
    try:
        local_path = source
        run_id = None
        
        # Handle raw Run ID (32 hex chars)
        if re.match(r'^[a-f0-9]{32}$', source):
            print(f"Detected Run ID: {source}. Searching for trade artifacts...")
            try:
                client = mlflow.MlflowClient()
                # Search in 'trades' folder and root
                potential_paths = ["trades", "generated_artifacts"]
                found_path = None
                
                for p in potential_paths:
                    try:
                        artifacts = client.list_artifacts(source, path=p)
                        for art in artifacts:
                            # Check for trades_*.csv
                            if os.path.basename(art.path).startswith("trades_") and art.path.endswith(".csv"):
                                found_path = art.path
                                break
                    except:
                        continue
                    if found_path: break
                
                if found_path:
                    source = f"runs:/{source}/{found_path}"
                    print(f"Auto-resolved artifact: {source}")
                else:
                    print(f"Warning: No 'trades_*.csv' found in run {source}")
            except Exception as e:
                print(f"Error searching artifacts for run {source}: {e}")

        # Handle MLflow artifacts
        if source.startswith("runs:/"):
            # Extract run_id
            parts = source.split('/')
            if len(parts) > 1:
                run_id = parts[1]
            print(f"Downloading artifact: {source}")
            local_path = mlflow.artifacts.download_artifacts(artifact_uri=source) # pyright: ignore[reportPrivateImportUsage]
            
        elif source.startswith("mlflow-artifacts:"):
             # Handle mlflow-artifacts scheme if necessary
             pass

        # Handle file URIs (e.g. file:///D:/...)
        elif source.startswith("file:"):
            parsed_url = urllib.parse.urlparse(source)
            local_path = urllib.parse.unquote(parsed_url.path)
            # On Windows, urlparse path might start with /D:/..., we need to strip leading /
            if os.name == 'nt' and local_path.startswith('/') and ':' in local_path:
                local_path = local_path.lstrip('/')
            
        # Try to extract run_id from path if not already found
        if not run_id:
            # Look for 32-char hex string inside 'mlruns' folder structure
            match = re.search(r'mlruns[\\/]([a-f0-9]{32})[\\/]', local_path)
            if match:
                run_id = match.group(1)

        # Default metadata
        ticker = "Unknown"
        timeframe = "Unknown"
        strategy = "Unknown"
        strategy_type = "Unknown"
        indicators = "Unknown"
        params_dict = {}
        
        # 1. Try to get metadata from MLflow
        if run_id:
            try:
                run = mlflow.get_run(run_id)
                tags = run.data.tags
                params = run.data.params
                
                if 'ticker' in tags: ticker = tags['ticker']
                if 'timeframe' in tags: timeframe = tags['timeframe']
                
                # Extract strategy metadata
                # User requested strategy to be [ticker]_[run name]
                run_name = tags.get('mlflow.runName')
                if run_name:
                    strategy = f"{run_name}"
                elif 'strategy_name' in params: 
                    strategy = params['strategy_name']
                
                strategy_type = params.get('strategy_type', 'single')
                
                if strategy_type == 'single':
                    indicators = params.get('indicator', 'Unknown')
                elif strategy_type == 'combo':
                    n_indicators = int(params.get('n_indicators', 0))
                    inds = []
                    for i in range(1, n_indicators + 1):
                        ind_name = params.get(f'ind{i}_name')
                        if ind_name:
                            inds.append(ind_name)
                    indicators = "+".join(inds)
                
                params_dict = params
                
                print(f"Loaded metadata from MLflow run {run_id}: {ticker} {timeframe} {strategy} ({strategy_type})")
            except Exception as e:
                print(f"Could not fetch run {run_id} from MLflow: {e}")
        
        # Fallback: Try to extract metadata from filename if MLflow failed or run_id not found
        if ticker == "Unknown":
            filename = os.path.basename(local_path)
            filename_no_ext = os.path.splitext(filename)[0]
            parts = filename_no_ext.split('_')
            
            if filename.startswith("trades_"):
                strategy = filename_no_ext[7:]
            
            if len(parts) >= 4:
                ticker = parts[1]
                timeframe = parts[2]
                strategy = "_".join(parts[3:])
            
        df = pd.read_csv(local_path)
        df['ticker'] = ticker
        df['timeframe'] = timeframe
        df['strategy'] = strategy
        df['strategy_type'] = strategy_type
        df['indicators'] = indicators
        df['source_file'] = os.path.basename(local_path)
        
        # Add params as a column (repeated for all rows)
        df['params'] = [params_dict for _ in range(len(df))]
        
        # Convert time columns
        if 'entry_time' in df.columns:
            df['entry_time'] = pd.to_datetime(df['entry_time'])
        if 'exit_time' in df.columns:
            df['exit_time'] = pd.to_datetime(df['exit_time'])
        
        # Convert duration
        if 'duration' in df.columns:
            df['duration_dt'] = pd.to_timedelta(df['duration'])
            df['duration_hours'] = df['duration_dt'].dt.total_seconds() / 3600 # pyright: ignore[reportAttributeAccessIssue]
        
        all_trades.append(df)
        
    except Exception as e:
        print(f"Error loading {source}: {e}")

if all_trades:
    trades_df = pd.concat(all_trades, ignore_index=True)
    print(f"Loaded {len(trades_df)} trades.")
    display(trades_df.head())
else:
    print("No trades loaded.")

MLflow Tracking URI: sqlite:///d:\py_projects\GammaNeutral\main\mlflow.db
Processing 8 trade sources.
Detected Run ID: 86a5affeeeb248b1b18a3a03e8d9781b. Searching for trade artifacts...
Auto-resolved artifact: runs:/86a5affeeeb248b1b18a3a03e8d9781b/generated_artifacts/trades_ethusd_CCI-TRIX-MACD.csv


Loaded metadata from MLflow run 86a5affeeeb248b1b18a3a03e8d9781b: ETHUSDT 1h ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo720wei2.0_WEIGHTED_L (combo)
Detected Run ID: ca3981e36ef94f859e090a6423fdd405. Searching for trade artifacts...
Auto-resolved artifact: runs:/ca3981e36ef94f859e090a6423fdd405/generated_artifacts/trades_ethusd_CCI-TRIX-MACD.csv


Loaded metadata from MLflow run ca3981e36ef94f859e090a6423fdd405: ETHUSDT 1h ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo720wei2.0_WEIGHTED_L (combo)
Detected Run ID: eabe859be4d64284955d36b4f2fe8738. Searching for trade artifacts...
Auto-resolved artifact: runs:/eabe859be4d64284955d36b4f2fe8738/generated_artifacts/trades_ethusd_CCI-TRIX-MACD.csv


Loaded metadata from MLflow run eabe859be4d64284955d36b4f2fe8738: ETHUSDT 1h ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo480wei2.0_WEIGHTED_L (combo)


Detected Run ID: 31b3d21e7b07478a98cd7198e2e051d7. Searching for trade artifacts...
Auto-resolved artifact: runs:/31b3d21e7b07478a98cd7198e2e051d7/generated_artifacts/trades_ethusd_CCI-TRIX-MACD.csv


...

Loaded metadata from MLflow run eb913bf9baf54fb69513a88759653bfa: ETHUSDT 1h ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo720wei2.0_WEIGHTED_B (combo)
Detected Run ID: 3548a88b870848028fd87ed9809dfe16. Searching for trade artifacts...
Auto-resolved artifact: runs:/3548a88b870848028fd87ed9809dfe16/generated_artifacts/trades_ethusd_CCI-TRIX-MACD.csv


Loaded metadata from MLflow run 3548a88b870848028fd87ed9809dfe16: ETHUSDT 1h ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo720wei2.0_WEIGHTED_B (combo)
Detected Run ID: 6fd43ddd3c094232bf70f4e3b7763fc6. Searching for trade artifacts...
Auto-resolved artifact: runs:/6fd43ddd3c094232bf70f4e3b7763fc6/generated_artifacts/trades_ethusd_CCI-TRIX-MACD.csv


Loaded metadata from MLflow run 6fd43ddd3c094232bf70f4e3b7763fc6: ETHUSDT 1h ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo480wei2.0_WEIGHTED_B (combo)
Detected Run ID: 42edb4d61c8a4e10af687b2eacc3dbfb. Searching for trade artifacts...
Auto-resolved artifact: runs:/42edb4d61c8a4e10af687b2eacc3dbfb/generated_artifacts/trades_ethusd_CCI-TRIX-MACD.csv


Loaded metadata from MLflow run 42edb4d61c8a4e10af687b2eacc3dbfb: ETHUSDT 1h ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15wei1.0-MACfas240sig15slo480wei2.0_WEIGHTED_B (combo)
Loaded 5829 trades.


,entry_time,exit_time,type,entry_price,exit_price,pnl_pct,duration,ticker,timeframe,strategy,strategy_type,indicators,source_file,params,duration_dt,duration_hours
0,2017-08-18 11:00:00,2017-08-18 17:00:00,long,308.88,297.50,-0.036843,0 days 06:00:00,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,combo,cci+trix+macd,trades_ethusd_CCI-TRIX-MACD.csv,"{'strategy_name': 'ethusd_CCI-TRIX-MACD', 'str...",0 days 06:00:00,6.0
1,2017-08-21 02:00:00,2017-09-02 10:00:00,long,299.00,351.60,0.175920,12 days 08:00:00,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,combo,cci+trix+macd,trades_ethusd_CCI-TRIX-MACD.csv,"{'strategy_name': 'ethusd_CCI-TRIX-MACD', 'str...",12 days 08:00:00,296.0
2,2017-09-06 10:00:00,2017-09-08 20:00:00,long,324.82,294.06,-0.094699,2 days 10:00:00,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,combo,cci+trix+macd,trades_ethusd_CCI-TRIX-MACD.csv,"{'strategy_name': 'ethusd_CCI-TRIX-MACD', 'str...",2 days 10:00:00,58.0
3,2017-09-11 23:00:00,2017-09-13 06:00:00,long,297.74,275.60,-0.074360,1 days 07:00:00,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,combo,cci+trix+macd,trades_ethusd_CCI-TRIX-MACD.csv,"{'strategy_name': 'ethusd_CCI-TRIX-MACD', 'str...",1 days 07:00:00,31.0
4,2017-09-16 12:00:00,2017-09-17 11:00:00,long,249.99,245.99,-0.016001,0 days 23:00:00,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,combo,cci+trix+macd,trades_ethusd_CCI-TRIX-MACD.csv,"{'strategy_name': 'ethusd_CCI-TRIX-MACD', 'str...",0 days 23:00:00,23.0


## 1.1 Detect Outliers
Identify statistical outliers in trade returns using the Interquartile Range (IQR) method.
Outliers are labeled in a new column `outlier` (True/False) to allow for easy filtering in subsequent analysis.

In [15]:
# 1.1 Detect Outliers
# We use the IQR method to label outliers in the 'pnl_pct' column.
# Outliers are calculated per strategy to account for different volatility profiles.

# is_outlier is imported from src.trading_strategy.utils.helpers

if not trades_df.empty:
    # Calculate outliers per (ticker, strategy) group
    # Using transform ensures the result aligns with the original DataFrame index
    trades_df['outlier'] = trades_df.groupby(['ticker', 'strategy'])['pnl_pct'].transform(is_outlier)
    
    n_outliers = trades_df['outlier'].sum()
    pct_outliers = n_outliers / len(trades_df)
    
    print(f"Outlier Detection (IQR=1.5):")
    print(f"  Total Trades: {len(trades_df)}")
    print(f"  Outliers: {n_outliers} ({pct_outliers:.2%})")
    
    if n_outliers > 0:
        print("\nTop 5 Positive Outliers:")
        display(trades_df[trades_df['outlier'] & (trades_df['pnl_pct'] > 0)]
                .sort_values('pnl_pct', ascending=False)[['ticker', 'strategy', 'entry_time', 'pnl_pct', 'outlier']].head())
        
        print("\nTop 5 Negative Outliers:")
        display(trades_df[trades_df['outlier'] & (trades_df['pnl_pct'] < 0)]
                .sort_values('pnl_pct', ascending=True)[['ticker', 'strategy', 'entry_time', 'pnl_pct', 'outlier']].head())
        
        # Visual check
        fig = px.strip(trades_df, x='outlier', y='pnl_pct', color='strategy', 
                       title='PnL % Distribution with Outliers Highlighted',
                       hover_data=['strategy', 'entry_time'],
                       width=1000, height=800
                       )
        fig.update_layout(legend=dict(
                                orientation="h",
                                yanchor="bottom",
                                y=-1.0,
                                xanchor="center",
                                x=0.5))
        fig.show()
else:
    print("No trades to analyze.")

Outlier Detection (IQR=1.5):
  Total Trades: 5829
  Outliers: 762 (13.07%)

Top 5 Positive Outliers:


,ticker,strategy,entry_time,pnl_pct,outlier
5267,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2020-12-25 13:00:00,0.714455,True
1666,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2020-12-25 13:00:00,0.714455,True
2005,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2018-04-02 02:00:00,0.671521,True
2990,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15w...,2018-04-02 02:00:00,0.671521,True
39,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2018-04-02 02:00:00,0.671521,True



Top 5 Negative Outliers:


,ticker,strategy,entry_time,pnl_pct,outlier
2056,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2018-10-15 04:00:00,-0.190691,True
5042,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2018-10-15 04:00:00,-0.190691,True
4029,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2018-10-15 04:00:00,-0.190691,True
3041,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15w...,2018-10-15 04:00:00,-0.190691,True
5062,ETHUSDT,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,2018-12-27 20:00:00,-0.167251,True


### Alternative Outlier Detection Methods
Besides IQR, other common methods include:
1. **Z-Score**: Marks data points that are a certain number of standard deviations away from the mean (usually > 3). Assumes normal distribution.
2. **Modified Z-Score**: Uses Median and Median Absolute Deviation (MAD). More robust to extreme values than standard Z-Score.
3. **Isolation Forest**: An unsupervised learning algorithm useful for high-dimensional data or complex distributions.

In [16]:
# 1.2 Alternative: Modified Z-Score
# This method is often more robust for financial data which may not be normally distributed.

# is_outlier_mod_zscore is imported from src.trading_strategy.utils.helpers

if not trades_df.empty:
    trades_df['outlier_mod_z'] = trades_df.groupby(['ticker', 'strategy'])['pnl_pct'].transform(is_outlier_mod_zscore)
    
    n_outliers_z = trades_df['outlier_mod_z'].sum()
    print(f"Modified Z-Score Detection (Threshold=3.5):")
    print(f"  Outliers: {n_outliers_z} ({n_outliers_z/len(trades_df):.2%})")
    
    # Compare overlap
    overlap = (trades_df['outlier'] & trades_df['outlier_mod_z']).sum()
    print(f"  Overlap with IQR method: {overlap}")
    
    # Visual check
    fig = px.strip(trades_df, x='outlier_mod_z', y='pnl_pct', color='strategy', 
                    title='PnL % Distribution with Outliers Z-score',
                    hover_data=['strategy', 'entry_time'],
                    width=1000, height=800
                    )
    fig.update_layout(legend=dict(
                            orientation="h",
                            yanchor="bottom",
                            y=-1.0,
                            xanchor="center",
                            x=0.5))
    fig.show()

Modified Z-Score Detection (Threshold=3.5):
  Outliers: 629 (10.79%)
  Overlap with IQR method: 629


## 2. Calculate Key Performance Metrics
Compute Win Rate, Profit Factor, Total Return, etc. for each strategy (Asset + Rank).

In [6]:
# calculate_trade_statistics is imported from src.trading_strategy.utils.helpers

def calculate_metrics_wrapper(group):
    # We use the imported function, specifying the returns column
    stats = calculate_trade_statistics(group, returns_col='pnl_pct')
    return pd.Series(stats)

# Group by strategy name instead of rank
metrics_by_strategy = trades_df.groupby(['ticker', 'timeframe', 'strategy']).apply(calculate_metrics_wrapper).reset_index()
display(metrics_by_strategy.sort_values('total_return', ascending=False))

C:\Users\soyel\AppData\Local\Temp\ipykernel_26708\536009932.py:9: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,ticker,timeframe,strategy,total_trades,winning_trades,losing_trades,win_rate,avg_win,avg_loss,largest_win,largest_loss,profit_factor,risk_reward,total_return,avg_return
1,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,986.0,330.0,656.0,0.334686,0.086071,-0.028243,0.671521,-0.190691,1.533032,3.047482,9.875801,0.010016
6,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15w...,993.0,332.0,661.0,0.334340,0.085838,-0.028212,0.671521,-0.190691,1.528199,3.042588,9.849997,0.009919
0,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,1021.0,342.0,679.0,0.334966,0.084322,-0.028759,0.659806,-0.190691,1.476819,2.932047,9.310917,0.009119
3,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,903.0,285.0,618.0,0.315615,0.093009,-0.029215,0.714455,-0.190691,1.468157,3.183583,8.452614,0.009361
2,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,494.0,171.0,323.0,0.346154,0.102540,-0.029796,0.671521,-0.147821,1.821931,3.441426,7.910331,0.016013
7,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15w...,498.0,170.0,328.0,0.341365,0.103516,-0.029553,0.671521,-0.147821,1.815438,3.502727,7.904369,0.015872
5,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp50wei1.0-TRIp25sig15w...,509.0,180.0,329.0,0.353635,0.098459,-0.030221,0.659806,-0.140587,1.782468,3.257956,7.779841,0.015285
4,ETHUSDT,1h,ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15w...,425.0,142.0,283.0,0.334118,0.114563,-0.032915,0.714455,-0.165663,1.746466,3.480633,6.953195,0.016360


## 3. Visualize Equity Curves
Plot cumulative returns over time for the top ranked strategies.

In [17]:
# Plot cumulative returns for ALL strategies

# Sort by total return (descending)
top_strategies = metrics_by_strategy.sort_values('total_return', ascending=False)

if top_strategies.empty:
    print("No strategies found.")
else:
    all_cumulative_data = []
    for _, row in top_strategies.iterrows(): # pyright: ignore[reportGeneralTypeIssues]
        mask = (trades_df['ticker'] == row['ticker']) & \
               (trades_df['timeframe'] == row['timeframe']) & \
               (trades_df['strategy'] == row['strategy'])
        
        strategy_trades = trades_df[mask].sort_values('exit_time').copy()
        strategy_trades['cumulative_return'] = strategy_trades['pnl_pct'].cumsum()
        strategy_trades['strategy_label'] = f"{row['strategy']}"
        all_cumulative_data.append(strategy_trades)
    
    if all_cumulative_data:
        plot_df = pd.concat(all_cumulative_data)
        fig = px.line(plot_df, x='exit_time', y='cumulative_return', color='strategy_label',
                      title='Cumulative Returns of All Strategies',
                      labels={'exit_time': 'Date', 'cumulative_return': 'Cumulative Return (Sum of PnL %)'},
                      width=1000, height=800)
        fig.update_layout(legend=dict(
                                orientation="h",
                                yanchor="bottom", y=-0.7,
                                xanchor="center", x=0.5))
        fig.show()

## 4. Analyze Trade Distribution and Duration
Examine the distribution of PnL and how trade duration relates to profitability.

In [20]:
# Filter out outliers if the column exists
plot_data = trades_df[trades_df['outlier'] == False] if 'outlier' in trades_df.columns else trades_df

# PnL Distribution
fig1 = px.histogram(plot_data, x='pnl_pct', color='strategy', nbins=50,
                    title='Distribution of Trade PnL % (Outliers Excluded)',
                    barmode='overlay',
                    width=1000, height=800)
fig1.update_layout(legend=dict(
                            orientation="h",
                            yanchor="bottom", y=-0.7,
                            xanchor="center", x=0.5))
fig1.show()

# Duration vs PnL
fig2 = px.scatter(plot_data, x='duration_hours', y='pnl_pct', color='strategy',
                  title='Trade Duration (Hours) vs PnL % (Outliers Excluded)',
                  opacity=0.2,
                  width=1000, height=800)
fig2.update_layout(legend=dict(
                            orientation="h",
                            yanchor="bottom", y=-0.7,
                            xanchor="center", x=0.5))
fig2.show()


## 5. Evaluate Drawdown Characteristics
Visualize the drawdown periods for the strategies.

In [9]:
def calculate_drawdown(pnl_series):
    cum_ret = pnl_series.cumsum()
    running_max = cum_ret.cummax()
    drawdown = cum_ret - running_max
    return drawdown

all_drawdown_data = []

for _, row in top_strategies.iterrows(): # pyright: ignore[reportGeneralTypeIssues]
    mask = (trades_df['ticker'] == row['ticker']) & \
           (trades_df['timeframe'] == row['timeframe']) & \
           (trades_df['strategy'] == row['strategy'])
    
    strategy_trades = trades_df[mask].sort_values('exit_time').copy()
    strategy_trades['drawdown'] = calculate_drawdown(strategy_trades['pnl_pct'])
    strategy_trades['strategy_label'] = f"{row['strategy']}"
    all_drawdown_data.append(strategy_trades)

if all_drawdown_data:
    plot_df = pd.concat(all_drawdown_data)
    fig = px.line(plot_df, x='exit_time', y='drawdown', color='strategy_label',
                  title='Drawdown Over Time',
                  labels={'exit_time': 'Date', 'drawdown': 'Drawdown (PnL %)'},
                  width=1000, height=800)
    fig.update_layout(legend=dict(
                            orientation="h",
                            yanchor="bottom", y=-0.7,
                            xanchor="center", x=0.5))
    fig.show()


## 6. Compare Strategy Performance by Asset
Compare Win Rate and Total Return across different assets.

In [10]:
# Compare metrics by asset
# Instead of filtering by 'rank1', we take the best performing strategy for each ticker
asset_metrics = metrics_by_strategy.sort_values('total_return', ascending=False).groupby('strategy').first().reset_index()

if asset_metrics.empty:
    print("No asset metrics to plot.")
else:
    fig1 = px.bar(asset_metrics, x='strategy', y='win_rate', title='Best Win Rate by Strategy')
    fig1.update_layout(legend=dict(
                                orientation="h",
                                yanchor="bottom", y=-0.7,
                                xanchor="center", x=0.5))
    fig1.show()

    fig2 = px.bar(asset_metrics, x='strategy', y='total_return', title='Best Total Return by Strategy')
    fig2.update_layout(legend=dict(
                                orientation="h",
                                yanchor="bottom", y=-0.7,
                                xanchor="center", x=0.5))
    fig2.show()


## 7. Analyze Win Rate vs. Risk-Reward Ratio
Scatter plot to categorize strategies based on their Win Rate and Risk/Reward profile.

In [11]:
metrics_by_strategy['risk_reward'] = abs(metrics_by_strategy['avg_win'] / metrics_by_strategy['avg_loss'])

fig = px.scatter(metrics_by_strategy, x='win_rate', y='risk_reward', symbol='ticker', color='strategy',
                 size_max=15, title='Win Rate vs Risk/Reward Ratio',
                 hover_data=['strategy', 'ticker'],
                 width=1000, height=800)
fig.update_layout(legend=dict(
                                orientation="h",
                                yanchor="bottom", y=-0.7,
                                xanchor="center", x=0.5))

# Add reference lines
fig.add_vline(x=0.5, line_dash="dash", line_color="gray")
fig.add_hline(y=1.0, line_dash="dash", line_color="gray")

fig.show()


## 8. Analysis of Losing Trades
Identify patterns in losing trades to refine the strategy.
- **Worst Trades**: Inspect the largest individual losses.
- **Seasonality**: Check if losses cluster around specific hours or days.
- **Duration**: See if losing trades tend to be shorter or longer than average.

In [21]:
# 8. Winning vs Losing Trade Analysis with Indicator Values

import sys
from pathlib import Path

# Add project root to path to import src modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.trading_strategy.data_loader import load_saved_data
from src.trading_strategy.indicators import calculate_indicator_and_signals, calculate_returns_and_momentum

trades_df_NoOutliers = trades_df[trades_df['outlier'] == False] if 'outlier' in trades_df.columns else trades_df

# Separate winners and losers
winning_trades = trades_df_NoOutliers[trades_df_NoOutliers['pnl_pct'] > 0].copy()
losing_trades = trades_df_NoOutliers[trades_df_NoOutliers['pnl_pct'] <= 0].copy()
print(f"Winning Trades: {len(winning_trades)} | Avg Win: {winning_trades['pnl_pct'].mean():.2%}")
print(f"Losing Trades: {len(losing_trades)} | Avg Loss: {losing_trades['pnl_pct'].mean():.2%}")

# --- 1. Duration Analysis ---
trades_df_NoOutliers['result'] = np.where(trades_df_NoOutliers['pnl_pct'] > 0, 'Win', 'Loss')
filtered_duration = trades_df_NoOutliers[trades_df_NoOutliers['duration_hours'] < trades_df_NoOutliers['duration_hours'].std()*3]
fig1 = px.box(filtered_duration, x='result', y='duration_hours', color='strategy',
              title='Trade Duration: Winners vs Losers',
              width=800, height=600)
fig1.update_layout(legend=dict(
                                orientation="h",
                                yanchor="bottom", y=-0.7,
                                xanchor="center", x=0.5))
fig1.show()

# --- 2. Time Analysis (Hour of Day) ---
trades_df_NoOutliers['hour'] = trades_df_NoOutliers['entry_time'].dt.hour
fig2 = px.histogram(trades_df_NoOutliers, x='hour', color='result', barmode='group',
                    title='Entry Hour Distribution: Winners vs Losers',
                    width=800, height=600)
fig2.update_layout(legend=dict(
                                orientation="h",
                                yanchor="bottom", y=-0.7,
                                xanchor="center", x=0.5))
fig2.show()

# --- 3. Indicator Value Analysis ---
# Recalculate indicators using metadata from MLflow (stored in trades_df)

# We will process each unique (ticker, timeframe, strategy) combination
# Include metadata columns. Note: 'params' is a dict (unhashable), so we can't use drop_duplicates on it directly.
unique_combos = trades_df_NoOutliers.groupby(['ticker', 'timeframe', 'strategy', 'strategy_type', 'indicators'])['params'].first().reset_index()

trades_with_indicators = []

print("Recalculating indicators for analysis...")

for _, row in unique_combos.iterrows(): # pyright: ignore[reportGeneralTypeIssues]
    ticker = row['ticker']
    timeframe = row['timeframe']
    strategy_name = row['strategy']
    strategy_type = row['strategy_type']
    indicators_str = row['indicators']
    params_raw = row['params']
    
    # 1. Load Market Data
    try:
        data = load_saved_data(ticker=ticker, timeframes=[timeframe])
        if not data or timeframe not in data:
            print(f"Skipping {ticker} {timeframe}: No market data found.")
            continue
        
        df_market = data[timeframe]
        # Ensure returns are calculated (needed for some indicators)
        df_market = calculate_returns_and_momentum(
            df_market,
            compute_indicators=False)
        
        # 2. Determine Indicators to Calculate
        indicators_to_calc = []
        
        if strategy_type == 'single':
            # Single strategy: indicators_str is the indicator name
            # params_raw contains the parameters directly
            
            # Convert params to correct types (int/float)
            clean_params = {}
            for k, v in params_raw.items():
                # Skip metadata keys
                if k in ['strategy_name', 'strategy_type', 'indicator', 'position_type', 'train_split', 'git_commit']:
                    continue
                try:
                    # Try to convert to number
                    if isinstance(v, str):
                        if '.' in v: clean_params[k] = float(v)
                        else: clean_params[k] = int(v)
                    else:
                        clean_params[k] = v
                except:
                    clean_params[k] = v
            
            indicators_to_calc.append((indicators_str, clean_params))
            
        elif strategy_type == 'combo':
            # Combo strategy: extract from ind{i}_* params
            # We look for ind1_name, ind2_name, etc.
            i = 1
            while True:
                ind_key = f"ind{i}_name"
                if ind_key not in params_raw:
                    break
                
                ind_name = params_raw[ind_key]
                ind_params = {}
                prefix = f"ind{i}_"
                
                for k, v in params_raw.items():
                    if k.startswith(prefix) and k != ind_key:
                        param_name = k[len(prefix):] # strip prefix
                        try:
                            if isinstance(v, str):
                                if '.' in v: ind_params[param_name] = float(v)
                                else: ind_params[param_name] = int(v)
                            else:
                                ind_params[param_name] = v
                        except:
                            ind_params[param_name] = v
                
                indicators_to_calc.append((ind_name, ind_params))
                i += 1
        
        if not indicators_to_calc:
            print(f"Skipping {strategy_name}: No indicators found in metadata.")
            continue

        # 3. Calculate Indicators
        for ind_name, ind_params in indicators_to_calc:
            try:
                # This adds columns to df_market (e.g., 'trix', 'trix_signal')
                df_market = calculate_indicator_and_signals(df_market, ind_name, ind_params, inplace=False)
                
                # Rename generic 'indicator_value' to specific indicator name if present
                if 'indicator_value' in df_market.columns:
                    df_market = df_market.rename(columns={'indicator_value': ind_name})
                    
            except Exception as e:
                print(f"  Error calculating {ind_name} for {strategy_name}: {e}")
        
        # 4. Merge with Trades
        # Filter trades for this specific combo
        mask = (trades_df['ticker'] == ticker) & \
               (trades_df['timeframe'] == timeframe) & \
               (trades_df['strategy'] == strategy_name)
        
        subset_trades = trades_df[mask].copy()
        
        # We need to join on entry_time. 
        # Market data index is timestamp.
        
        # Get list of indicator columns (those added to df_market)
        # We know standard cols, so new ones are indicators
        standard_market_cols = ['open', 'high', 'low', 'close', 'volume', 'returns', 'log_returns', 'volatility', 'momentum']
        indicator_cols = [c for c in df_market.columns if c not in standard_market_cols and c != 'signal']
        
        # Create a lookup df
        lookup_df = df_market[indicator_cols]
        
        # Merge
        subset_trades = subset_trades.merge(lookup_df, left_on='entry_time', right_index=True, how='left')
        
        trades_with_indicators.append(subset_trades)
        print(f"Processed {ticker} {strategy_name}: Added {indicator_cols}")
        
    except Exception as e:
        print(f"Error processing {ticker} {strategy_name}: {e}")

if trades_with_indicators:
    full_trades_df = pd.concat(trades_with_indicators, ignore_index=True)
    full_trades_df_NoOutliers = full_trades_df[full_trades_df['outlier'] == False] if 'outlier' in full_trades_df.columns else full_trades_df
    
    # Now plot indicator values for Winners vs Losers
    # Identify numeric columns that are not standard trade cols
    std_cols = set(trades_df_NoOutliers.columns)
    new_cols = [c for c in full_trades_df_NoOutliers.columns if c not in std_cols and pd.api.types.is_numeric_dtype(full_trades_df_NoOutliers[c])]
    
    if new_cols:
        print(f"\nAnalyzing Indicator Values: {new_cols}\n")
        for col in new_cols:
            fig = px.histogram(
                full_trades_df_NoOutliers, x=col, y='pnl_pct', color='strategy',
                title=f'{col} vs pnl_pct',
                barmode='overlay', opacity=0.6,
                hover_data=['strategy', col],
                histfunc='avg',
                width=800, height=600)
            fig.update_layout(legend=dict(
                                    orientation="h",
                                    yanchor="bottom", y=-0.7,
                                    xanchor="center", x=0.5))
            fig.show()
    else:
        print("No indicator columns were successfully added.")
else:
    print("Could not recalculate indicators for any strategy.")


Winning Trades: 1264 | Avg Win: 2.78%
Losing Trades: 3803 | Avg Loss: -2.73%


Recalculating indicators for analysis...
📂 Cargando datos guardados de ETHUSDT...
  ✓ 1h: 72107 velas (2017-08-17 04:00:00 a 2025-11-12 22:00:00)
✓ Datos cargados exitosamente desde D:\py_projects\GammaNeutral\main\data\market/
Processed ETHUSDT ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo480wei2.0_WEIGHTED_B: Added ['Open', 'High', 'Low', 'Close', 'Volume', 'candle_rtn', 'future_ret+1', 'cci', 'trix', 'macd', 'macd_signal_line', 'macd_histogram']
📂 Cargando datos guardados de ETHUSDT...
  ✓ 1h: 72107 velas (2017-08-17 04:00:00 a 2025-11-12 22:00:00)
✓ Datos cargados exitosamente desde D:\py_projects\GammaNeutral\main\data\market/
Processed ETHUSDT ethusd_CCI-TRIX-MACD_CCIp30wei1.0-TRIp25sig15wei1.0-MACfas120sig15slo720wei2.0_WEIGHTED_B: Added ['Open', 'High', 'Low', 'Close', 'Volume', 'candle_rtn', 'future_ret+1', 'cci', 'trix', 'macd', 'macd_signal_line', 'macd_histogram']
📂 Cargando datos guardados de ETHUSDT...
  ✓ 1h: 72107 velas (2017-08-17 04:00:00 a 202